In [1]:
!uv pip install pymupdf pypdf langchain_community

Using Python 3.12.11 environment at: /Users/sungyonglee/github/do_it_ai_agent_2026/.venv
Audited 3 packages in 25ms


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/OneNYC_2050_Strategic_Plan.pdf")

data_nyc = loader.load()
print(data_nyc)

/var/folders/rl/fl_699v15019f3lzfwnb4zjr0000gn/T/ipykernel_11272/2143093783.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자 단위로 나눔. overlap은 100자로 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_splits = text_splitter.split_documents(data_nyc)

In [ ]:
for i, split in enumerate(all_splits):
    print(f"Split {i+1}:------------------------------------\n")
    print(split)

In [ ]:
print(type(all_splits[0]))

In [ ]:
loader_seoul = PyPDFLoader("../data/2040_seoul_plan.pdf")
data_seoul = loader_seoul.load()
seoul_splits = text_splitter.split_documents(data_seoul)
for i, split in enumerate(seoul_splits):
    print(f"Split {i+1}:------------------------------------\n")
    print(split)

In [ ]:
print(seoul_splits[50].page_content)
print('----------------------')
print(seoul_splits[51].page_content)

In [ ]:
for i in range(len(seoul_splits) - 1):
    seoul_splits[i].page_content += "\n"+ seoul_splits[i + 1].page_content[:100]

print(seoul_splits[50].page_content)
print('----------------------')
print(seoul_splits[51].page_content)

In [ ]:
print(len(all_splits))
all_splits.extend(seoul_splits)
print(len(all_splits))

In [ ]:
!uv pip install chromadb

In [ ]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)
v = embedding.embed_query("뉴욕의 온실가스 저감 정책은 뭐야?")
print(v)
print(len(v))

In [ ]:
!uv pip install langchain_chroma

In [ ]:
from langchain_chroma import Chroma
import os

persist_directory = "../chroma_store"

if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )
else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding
    )

In [ ]:
retriever = vectorstore.as_retriever(k=3)
docs = retriever.invoke("서울시의 환경 정책이 궁금해.")

for d in docs:
    print(d)
    print('------')

In [ ]:
def retrieve_docs(query):
    """
    vector store에서 query에 대한 관련 문서를 검색하고, 검색된 문서들을 직렬화하여 반환하는 함수
    """
    retriever = vectorstore.as_retriever(k=3)
    docs = retriever.invoke(query)
    
    serialized_docs = "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}" for doc in docs
    )

    return serialized_docs

retrieve_docs("서울시의 환경 정책이 궁금해.")

In [ ]:
from langchain.agents import create_agent

system_prompt = """
너는 도시 정책 전문가로서 사용자의 질문에 대해 주어진 자료에 기반하여 대답해야 한다.
도시 정책에 관한 질문을 받았을 때 'retriever'를 사용하여 관련 자료를 검색하고, 그 자료를 바탕으로 답변을 작성해야 한다.
근거없는 추측이나 의견은 피하고, 반드시 자료에 기반한 사실만을 제공해야 한다.
"""

agent = create_agent(
    model="openai:gpt-5.4-mini",
    system_prompt=system_prompt,
    tools=[retrieve_docs]
)

response = agent.invoke({
    "messages": "서울시의 환경 정책이 궁금해"
})

for m in response['messages']:
    m.pretty_print()

In [ ]:
from langchain.messages import AIMessageChunk

agent = create_agent(
    model="openai:gpt-5.4-mini",
    system_prompt=system_prompt,
    tools=[retrieve_docs]
)

sources = []

for chunk in agent.stream(
    {"messages": "서울시의 환경 정책이 궁금해"},
    stream_mode=["messages"],
    version="v2"
):
    

    if chunk["type"] == "messages":
        token, metadata = chunk["data"]

        if isinstance(token, AIMessageChunk):
        # 일반 텍스트 토큰만 화면에 표시
            print(token.text, end="", flush=True)

In [ ]:
sources = []

for chunk in agent.stream(
    {"messages": "서울시의 환경 정책이 궁금해"},
    stream_mode=["messages", "updates"],
    version="v2"
):
    

    if chunk["type"] == "messages":
        token, metadata = chunk["data"]

        if isinstance(token, AIMessageChunk):
        # 일반 텍스트 토큰만 화면에 표시
            print(token.text, end="", flush=True)
            
    elif chunk["type"] == "updates":
        if "tools" in chunk["data"]:
            tools = chunk["data"]["tools"]["messages"]

            for t in tools:
                sources.append(t.content)
                print("TOOL\n", t.content)
        print("\n----------------")
    

In [ ]:
from langchain.messages import AIMessageChunk

sources = []

for chunk in agent.stream(
    {"messages": "서울시의 환경 정책이 궁금해"},
    stream_mode=["messages", "values", "updates"],
    version="v2"
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]

        if isinstance(token, AIMessageChunk):
        # 일반 텍스트 토큰만 화면에 표시
            print(token.text, end="", flush=True)
            # if token.text:
            #     streamed_text += token.text
            #     response_box.markdown(streamed_text)
    elif chunk["type"] == "updates":
        if "tools" in chunk["data"]:
            tools = chunk["data"]["tools"]["messages"]

            for t in tools:
                sources.append(t.content)
    
    elif chunk["type"] == "values":
        print(
            "\n------ values ------ message length", 
            len(chunk["data"]["messages"]),
            type(chunk["data"]["messages"][-1])
        )
        messages = chunk["data"]["messages"]

            
    
print("\n------ sources ------ ")
        
for s in sources:
    print(s)


print("\n------ messages ------ ")
for m in messages:
    m.pretty_print()